In [1]:
import os
import pandas as pd
import subprocess

In [2]:
SUBMISSION_FOLDER = "your_submission_data_development"

In [3]:
result_files = [f for f in os.listdir(SUBMISSION_FOLDER) if f.endswith(".jsonl") and "mock" not in f and "example" not in f and "dummy" not in f]
result_files

['gpt-5-nano_en.jsonl',
 'gpt-oss-120b_new-prompt_it.jsonl',
 'gemini-3-flash-preview_new-prompt_en.jsonl',
 'llama-4-maverick-17b-128e-instruct_re-elaborating-gpt-oss-120b_en.jsonl',
 'llama-4-maverick-17b-128e-instruct_re-elaborating-qwen3-32b_new-prompt_en.jsonl',
 '[best-en, 0.612]llama-4-maverick-17b-128e-instruct_re-elaborating-gemini-3-flash-preview_new-prompt_en.jsonl',
 'gemma-3-27b-it_dynamic-temp_en.jsonl',
 'gpt-oss-120b_re-elaborating-gemini-3-flash-preview_it.jsonl',
 'gpt-5-mini_en.jsonl',
 '[best-it, 0.599]gemini-3-flash-preview_new-prompt_it.jsonl',
 'llama-4-maverick-17b-128e-instruct_first-attempt_en.jsonl',
 'gpt-oss-120b_re-elaborating-gemini-3-flash-preview_en.jsonl',
 'llama-4-maverick-17b-128e-instruct_re-elaborating-for-2nd-time-gemini-3-flash-preview_new-prompt_en.jsonl',
 'qwen3-32b_dynamic-temp_en.jsonl',
 'gemini-3-flash-preview_en.jsonl',
 'llama-4-maverick-17b-128e-instruct_retrying-on-errors_en.jsonl',
 'gpt-5-nano_re-elaborating-gemini-3-flash-preview_e

In [4]:
def run_scoring(f):
    submission_path = os.path.join(SUBMISSION_FOLDER, f)
    language = f.split(".jsonl")[0][-2:]

    cmd = [
        "python3",
        "scoring.py",
        "--submission_path", submission_path,
        "--language", language
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    out = result.stdout
    try:
        TP = out.split("TP: ")[1].split(",")[0]
        FP = out.split("FP: ")[1].split(",")[0]
        FN = out.split("FN: ")[1].split("\n")[0]
        F1 = out.split("Final macro-F1 = ")[1].split("\n")[0]
    except:
        print(out, result.stderr)
        raise "Errore"
    return TP, FP, FN, F1

In [ ]:
# filter only first step
# result_files = [f for f in result_files if "re-elaborating" not in f]

In [7]:
computed_scorings = {f: run_scoring(f) for f in result_files}

In [12]:
df = pd.DataFrame(
    [(k, *v) for (k, v) in computed_scorings.items()],
    columns=["name", "TP", "FP", "FN", "f1-score"]
)
df["f1-score"] = df["f1-score"].astype(float).round(3)
df["language"] = df["name"].str.split(".jsonl").str[0].str[-2:]
df.index = df["name"]
df.drop("name", axis=1, inplace=True)
df.sort_values(by="f1-score", ascending=False, inplace=True)
df

,TP,FP,FN,f1-score,language
name,,,,,
gpt-5-nano_en.jsonl,237,160,115,0.604,en
gpt-oss-120b_dynamic-temp_en.jsonl,260,239,90,0.604,en
"[best-it, 0.599]gemini-3-flash-preview_new-prompt_it.jsonl",289,385,56,0.600,it
llama-4-maverick-17b-128e-instruct_27jan_en.jsonl,111,83,130,0.599,en
gpt-5-mini_en.jsonl,283,257,56,0.597,en
gemma-3-27b-it_dynamic-temp_en.jsonl,190,207,175,0.585,en
gemini-3-flash-preview_new-prompt_en.jsonl,278,420,83,0.584,en
gpt-5-nano_new-prompt_it.jsonl,207,140,145,0.582,it
llama-4-maverick-17b-128e-instruct_dynamic-temp_en.jsonl,179,131,170,0.579,en
